# Random Forest

Standard tabular ML workflow for LD50 regression.


In [4]:
from pathlib import Path
import sys
import warnings

import numpy as np
from sklearn.ensemble import RandomForestRegressor

PROJECT_ROOT = Path("../..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.modeling import artifact_paths, load_tabular_arrays, save_ml_run


In [5]:
MODEL_NAME = "random_forest"
BASE_DIR = Path.cwd()
RANDOM_STATE = 42

X_train, X_valid, X_test, y_train, y_valid, y_test, feature_names = load_tabular_arrays('../../data/feature_selection')
X_train = X_train.astype(np.float32)
X_valid = X_valid.astype(np.float32)
X_test = X_test.astype(np.float32)

paths = artifact_paths(
    BASE_DIR,
    MODEL_NAME,
    model_suffix=".pkl",
    include_importance=True,
)


In [6]:
try:
    from cuml.ensemble import RandomForestRegressor as CumlRandomForest

    model = CumlRandomForest(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        random_state=RANDOM_STATE,
    )
except Exception:
    warnings.warn("cuML RandomForestRegressor not available; using sklearn on CPU.")
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

model.fit(X_train, y_train)
predictions = model.predict(X_test)

C:\Users\Tommaso\AppData\Local\Temp\ipykernel_35072\570299048.py:11: UserWarning: cuML RandomForestRegressor not available; using sklearn on CPU.
  warnings.warn("cuML RandomForestRegressor not available; using sklearn on CPU.")


In [7]:
metrics = save_ml_run("Random Forest", model, paths, X_test, y_test, predictions, feature_names)

[Random Forest] RMSE: 0.5991 | MAE: 0.4385 | R2: 0.5983
